[← Back to Clinical Overview](../01_clinical_new.ipynb)

# TCGA-BRCA Comprehensive Demographics & Diagnosis Analysis

This notebook provides comprehensive exploratory data analysis (EDA) of all demographic and diagnosis-related columns using interactive Bokeh visualizations.

**Analysis Coverage:**
- All 30 diagnosis-related columns
- Temporal patterns (diagnosis timing, age distributions)
- Staging and classification systems
- Treatment patterns
- Cross-variable comparisons and correlations

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Import libraries
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

# Bokeh imports
from bokeh.plotting import figure, show, output_notebook
from bokeh.layouts import gridplot, column, row
from bokeh.models import HoverTool, ColumnDataSource, Legend, LegendItem
from bokeh.transform import cumsum, factor_cmap
from bokeh.palettes import Category20, Viridis256, Spectral11, Turbo256
from bokeh.io import export_png
import colorcet as cc

# Additional utilities
from oncolearn.api.xenabrowser import XenaCohortBuilder

warnings.filterwarnings('ignore')

# Enable Bokeh in notebook
output_notebook()

Loading BokehJS ...

## Load Clinical Data

Using the `XenaCohort` class to load all clinical datasets.

In [3]:
# Build the cohort
builder = XenaCohortBuilder()
cohort = builder.build_cohort("BRCA")

# Load clinical data
clinical = cohort.clinical()

Deduplicated brca/pam50: 141 -> 74 rows


## Extract Demographic/Diagnosis Columns

We'll focus on the 30 diagnosis-related columns specified:

In [4]:
# Define all demographic columns of interest
diagnosis_columns = [col for col in clinical.columns if 'diagnoses' in col]

# Check which columns exist in the dataset
existing_cols = [col for col in diagnosis_columns if col in clinical.columns]
missing_cols = [
    col for col in diagnosis_columns if col not in clinical.columns]

print(f"Found {len(existing_cols)} out of {len(diagnosis_columns)} columns")
print(f"\n✓ Available columns: {len(existing_cols)}")
print(f"✗ Missing columns: {len(missing_cols)}")

if missing_cols:
    print("\nMissing columns:")
    for col in missing_cols:
        print(f"  - {col}")

# Create subset DataFrame
demo_df = clinical[existing_cols].copy()
print(f"\nDemographic subset shape: {demo_df.shape}")

Found 30 out of 30 columns

✓ Available columns: 30
✗ Missing columns: 0

Demographic subset shape: (1257, 30)


---
## Part 1: Data Quality Assessment

Understanding missing data patterns across all demographic variables.

In [5]:
# Calculate missing data statistics
missing_stats = pd.DataFrame({
    'column': demo_df.columns,
    'missing_count': demo_df.isnull().sum(),
    'missing_pct': (demo_df.isnull().sum() / len(demo_df)) * 100,
    'present_count': demo_df.notnull().sum(),
    'unique_values': [demo_df[col].nunique() for col in demo_df.columns]
})

missing_stats = missing_stats.sort_values('missing_pct', ascending=False)

print("Data Quality Summary:")
print("="*80)
print(f"Total patients: {len(demo_df):,}")
print(f"Total columns: {len(demo_df.columns)}")
print(f"\nColumns by completeness:")
print(
    f"  Complete (0% missing):        {len(missing_stats[missing_stats['missing_pct'] == 0]):3d}")
print(
    f"  High quality (<10% missing):  {len(missing_stats[missing_stats['missing_pct'] < 10]):3d}")
print(
    f"  Medium quality (10-50%):      {len(missing_stats[(missing_stats['missing_pct'] >= 10) & (missing_stats['missing_pct'] < 50)]):3d}")
print(
    f"  Low quality (>50% missing):   {len(missing_stats[missing_stats['missing_pct'] >= 50]):3d}")

print("\n" + "="*80)
print("Top 10 columns with most missing data:")
print(missing_stats.head(10).to_string(index=False))

Data Quality Summary:
Total patients: 1,257
Total columns: 30

Columns by completeness:
  Complete (0% missing):          0
  High quality (<10% missing):   28
  Medium quality (10-50%):        2
  Low quality (>50% missing):     0

Top 10 columns with most missing data:
                                                   column  missing_count  missing_pct  present_count  unique_values
                    ajcc_staging_system_edition.diagnoses            177    14.081146           1080              5
                         days_to_last_follow_up.diagnoses            150    11.933174           1107            762
                               age_at_diagnosis.diagnoses             19     1.511535           1238           1044
         age_at_earliest_diagnosis.diagnoses.xena_derived             19     1.511535           1238           1044
age_at_earliest_diagnosis_in_years.diagnoses.xena_derived             19     1.511535           1238           1044
                          ajcc_p

In [6]:
# Bokeh visualization: Missing data heatmap-style bar chart
from bokeh.models import FactorRange

# Prepare data
sorted_stats = missing_stats.sort_values('missing_pct', ascending=True)
source = ColumnDataSource(sorted_stats)

# Shorten column names for display
short_names = [col.replace('.diagnoses', '').replace('.xena_derived', '').replace('.treatments', '')
               for col in sorted_stats['column']]

# Create figure
p = figure(
    y_range=short_names,
    height=max(600, len(short_names) * 20),
    width=900,
    title="Data Completeness Across All Demographic Columns",
    toolbar_location="right",
    tools="pan,wheel_zoom,box_zoom,reset,save"
)

# Color map based on missing percentage
colors = []
for pct in sorted_stats['missing_pct']:
    if pct == 0:
        colors.append('#2ecc71')  # Green - complete
    elif pct < 10:
        colors.append('#3498db')  # Blue - high quality
    elif pct < 50:
        colors.append('#f39c12')  # Orange - medium
    else:
        colors.append('#e74c3c')  # Red - low quality

# Add bars
p.hbar(
    y=short_names,
    right=sorted_stats['missing_pct'],
    height=0.8,
    color=colors,
    alpha=0.8
)

# Styling
p.xaxis.axis_label = "Missing Data (%)"
p.yaxis.axis_label = "Column"
p.xgrid.grid_line_color = None
p.ygrid.grid_line_color = "#dddddd"

# Add hover tool
hover = HoverTool(tooltips=[
    ("Column", "@column"),
    ("Missing", "@missing_count (@missing_pct{0.1f}%)"),
    ("Present", "@present_count"),
    ("Unique Values", "@unique_values")
])
p.add_tools(hover)

show(p)

---
## Part 2: Age-Related Analysis

Comprehensive analysis of age at diagnosis and temporal patterns.

In [7]:
# Age distribution analysis
age_col = 'age_at_diagnosis.diagnoses'
age_earliest_col = 'age_at_earliest_diagnosis_in_years.diagnoses.xena_derived'

# Convert to numeric
age_data = pd.to_numeric(demo_df[age_col], errors='coerce').dropna()
age_earliest_data = pd.to_numeric(demo_df[age_earliest_col], errors='coerce').dropna(
) if age_earliest_col in demo_df.columns else None

print(f"Age at Diagnosis Statistics:")
print(f"  Count: {len(age_data):,}")
print(f"  Mean: {age_data.mean():.1f} years")
print(f"  Median: {age_data.median():.1f} years")
print(f"  Std Dev: {age_data.std():.1f} years")
print(f"  Range: {age_data.min():.0f} - {age_data.max():.0f} years")
print(f"  Q1 (25%): {age_data.quantile(0.25):.1f} years")
print(f"  Q3 (75%): {age_data.quantile(0.75):.1f} years")
print(f"  IQR: {age_data.quantile(0.75) - age_data.quantile(0.25):.1f} years")

if age_earliest_data is not None:
    print(f"\nAge at Earliest Diagnosis Statistics:")
    print(f"  Count: {len(age_earliest_data):,}")
    print(f"  Mean: {age_earliest_data.mean():.1f} years")
    print(f"  Median: {age_earliest_data.median():.1f} years")

Age at Diagnosis Statistics:
  Count: 1,238
  Mean: 21489.4 years
  Median: 21472.0 years
  Std Dev: 4865.0 years
  Range: 9706 - 32872 years
  Q1 (25%): 17786.0 years
  Q3 (75%): 24791.0 years
  IQR: 7005.0 years

Age at Earliest Diagnosis Statistics:
  Count: 1,238
  Mean: 58.9 years
  Median: 58.8 years


In [8]:
# Bokeh: Age distribution histogram with KDE overlay
from scipy import stats

# Create histogram data
hist, edges = np.histogram(age_data, bins=40, density=True)
hist_df = pd.DataFrame({
    'top': hist,
    'left': edges[:-1],
    'right': edges[1:]
})

# KDE for smooth curve
kde = stats.gaussian_kde(age_data)
x_range = np.linspace(age_data.min(), age_data.max(), 200)
kde_values = kde(x_range)

# Create figure
p1 = figure(
    width=900,
    height=400,
    title="Age at Diagnosis Distribution",
    tools="pan,wheel_zoom,box_zoom,reset,save",
    x_axis_label="Age (years)",
    y_axis_label="Density"
)

# Add histogram
p1.quad(
    top='top',
    bottom=0,
    left='left',
    right='right',
    source=ColumnDataSource(hist_df),
    fill_color='#3498db',
    line_color='white',
    alpha=0.7,
    legend_label='Histogram'
)

# Add KDE curve
p1.line(x_range, kde_values, line_width=3, color='#e74c3c',
        alpha=0.8, legend_label='KDE')

# Add vertical lines for statistics
p1.line([age_data.mean(), age_data.mean()], [0, max(hist)],
        line_width=2, color='orange', line_dash='dashed',
        legend_label=f'Mean: {age_data.mean():.1f}')
p1.line([age_data.median(), age_data.median()], [0, max(hist)],
        line_width=2, color='green', line_dash='dashed',
        legend_label=f'Median: {age_data.median():.1f}')

p1.legend.location = "top_right"
p1.legend.click_policy = "hide"

show(p1)

In [9]:
# Box plot for age groups
from bokeh.models import Label
age_groups = pd.cut(age_data, bins=[0, 40, 50, 60, 70, 100],
                    labels=['<40', '40-50', '50-60', '60-70', '70+'])
age_group_counts = age_groups.value_counts().sort_index()

# Prepare data for Bokeh
group_names = [str(x) for x in age_group_counts.index]
counts = age_group_counts.values

source = ColumnDataSource(data=dict(
    groups=group_names,
    counts=counts,
    percentages=[f"{(c/len(age_data)*100):.1f}%" for c in counts]
))

p2 = figure(
    x_range=group_names,
    width=900,
    height=400,
    title="Age Groups Distribution at Diagnosis",
    toolbar_location="right",
    tools="pan,wheel_zoom,box_zoom,reset,save"
)

p2.vbar(
    x='groups',
    top='counts',
    width=0.8,
    source=source,
    fill_color=factor_cmap('groups', palette=Spectral11, factors=group_names),
    line_color='white',
    alpha=0.8
)

# Add value labels on bars
for i, (group, count, pct) in enumerate(zip(group_names, counts, source.data['percentages'])):
    label = Label(x=i, y=count, text=f"{count}\n({pct})",
                  text_align='center', text_baseline='bottom',
                  text_font_size='10pt')
    p2.add_layout(label)

p2.xaxis.axis_label = "Age Group"
p2.yaxis.axis_label = "Number of Patients"

hover = HoverTool(tooltips=[
    ("Age Group", "@groups"),
    ("Count", "@counts"),
    ("Percentage", "@percentages")
])
p2.add_tools(hover)

show(p2)

---
## Part 3: Temporal Analysis

Analysis of diagnosis timing patterns.

In [10]:
# Year of diagnosis analysis
year_col = 'year_of_diagnosis.diagnoses'
days_to_dx_col = 'days_to_diagnosis.diagnoses'

if year_col in demo_df.columns:
    year_data = pd.to_numeric(demo_df[year_col], errors='coerce').dropna()

    print("Year of Diagnosis Distribution:")
    print(f"  Range: {year_data.min():.0f} - {year_data.max():.0f}")
    print(f"  Median: {year_data.median():.0f}")
    print(f"  Mode: {year_data.mode().values[0]:.0f}")

    # Year distribution
    year_counts = year_data.value_counts().sort_index()

    p3 = figure(
        width=900,
        height=400,
        title="Number of Diagnoses by Year",
        tools="pan,wheel_zoom,box_zoom,reset,save",
        x_axis_label="Year",
        y_axis_label="Number of Diagnoses"
    )

    p3.line(year_counts.index, year_counts.values,
            line_width=2, color='#3498db', alpha=0.8)
    p3.circle(year_counts.index, year_counts.values,
              size=6, color='#e74c3c', alpha=0.6)

    hover = HoverTool(tooltips=[
        ("Year", "$x{0}"),
        ("Count", "$y")
    ])
    p3.add_tools(hover)

    show(p3)

Year of Diagnosis Distribution:
  Range: 1988 - 2013
  Median: 2009
  Mode: 2010


In [11]:
# Days to diagnosis and follow-up
days_to_dx = pd.to_numeric(demo_df[days_to_dx_col], errors='coerce').dropna(
) if days_to_dx_col in demo_df.columns else None
days_to_fu_col = 'days_to_last_follow_up.diagnoses'
days_to_fu = pd.to_numeric(demo_df[days_to_fu_col], errors='coerce').dropna(
) if days_to_fu_col in demo_df.columns else None

if days_to_dx is not None and len(days_to_dx) > 0:
    print(f"\nDays to Diagnosis:")
    print(f"  Count: {len(days_to_dx):,}")
    print(f"  Mean: {days_to_dx.mean():.1f} days")
    print(f"  Median: {days_to_dx.median():.1f} days")

if days_to_fu is not None and len(days_to_fu) > 0:
    print(f"\nDays to Last Follow-up:")
    print(f"  Count: {len(days_to_fu):,}")
    print(
        f"  Mean: {days_to_fu.mean():.1f} days ({days_to_fu.mean()/365.25:.1f} years)")
    print(
        f"  Median: {days_to_fu.median():.1f} days ({days_to_fu.median()/365.25:.1f} years)")
    print(f"  Range: {days_to_fu.min():.0f} - {days_to_fu.max():.0f} days")

    # Histogram of follow-up time
    fu_years = days_to_fu / 365.25
    hist_fu, edges_fu = np.histogram(fu_years, bins=50)

    p4 = figure(
        width=900,
        height=400,
        title="Follow-up Time Distribution",
        tools="pan,wheel_zoom,box_zoom,reset,save",
        x_axis_label="Follow-up Time (years)",
        y_axis_label="Number of Patients"
    )

    p4.quad(top=hist_fu, bottom=0, left=edges_fu[:-1], right=edges_fu[1:],
            fill_color='#9b59b6', line_color='white', alpha=0.7)

    # Add median line
    p4.line([fu_years.median(), fu_years.median()], [0, max(hist_fu)],
            line_width=2, color='red', line_dash='dashed',
            legend_label=f'Median: {fu_years.median():.1f} years')

    p4.legend.location = "top_right"

    show(p4)


Days to Diagnosis:
  Count: 1,254
  Mean: 0.0 days
  Median: 0.0 days

Days to Last Follow-up:
  Count: 1,107
  Mean: 1184.8 days (3.2 years)
  Median: 788.0 days (2.2 years)
  Range: -7 - 8605 days


---
## Part 4: AJCC Staging System

Comprehensive analysis of AJCC pathologic staging components.

In [12]:
# AJCC staging columns
staging_cols = {
    'Overall Stage': 'ajcc_pathologic_stage.diagnoses',
    'T Stage': 'ajcc_pathologic_t.diagnoses',
    'N Stage': 'ajcc_pathologic_n.diagnoses',
    'M Stage': 'ajcc_pathologic_m.diagnoses',
    'Staging Edition': 'ajcc_staging_system_edition.diagnoses'
}

# Display distributions
for name, col in staging_cols.items():
    if col in demo_df.columns:
        value_counts = demo_df[col].value_counts()
        print(f"\n{name} ({col}):")
        print(f"  Unique values: {len(value_counts)}")
        print(
            f"  Missing: {demo_df[col].isnull().sum()} ({demo_df[col].isnull().sum()/len(demo_df)*100:.1f}%)")
        print(f"  Top 5 values:")
        for val, count in value_counts.head(5).items():
            print(f"    {val}: {count} ({count/len(demo_df)*100:.1f}%)")


Overall Stage (ajcc_pathologic_stage.diagnoses):
  Unique values: 12
  Missing: 15 (1.2%)
  Top 5 values:
    Stage IIA: 409 (32.5%)
    Stage IIB: 303 (24.1%)
    Stage IIIA: 174 (13.8%)
    Stage I: 111 (8.8%)
    Stage IA: 93 (7.4%)

T Stage (ajcc_pathologic_t.diagnoses):
  Unique values: 13
  Missing: 3 (0.2%)
  Top 5 values:
    T2: 727 (57.8%)
    T1c: 255 (20.3%)
    T3: 152 (12.1%)
    T1: 46 (3.7%)
    T4b: 34 (2.7%)

N Stage (ajcc_pathologic_n.diagnoses):
  Unique values: 16
  Missing: 3 (0.2%)
  Top 5 values:
    N0: 386 (30.7%)
    N1a: 192 (15.3%)
    N0 (i-): 170 (13.5%)
    N1: 144 (11.5%)
    N2a: 70 (5.6%)

M Stage (ajcc_pathologic_m.diagnoses):
  Unique values: 4
  Missing: 3 (0.2%)
  Top 5 values:
    M0: 1043 (83.0%)
    MX: 180 (14.3%)
    M1: 24 (1.9%)
    cM0 (i+): 7 (0.6%)

Staging Edition (ajcc_staging_system_edition.diagnoses):
  Unique values: 5
  Missing: 177 (14.1%)
  Top 5 values:
    6th: 494 (39.3%)
    7th: 457 (36.4%)
    5th: 92 (7.3%)
    4th: 29 (2

In [13]:
# Visualize AJCC Overall Stage
stage_col = 'ajcc_pathologic_stage.diagnoses'

if stage_col in demo_df.columns:
    stage_counts = demo_df[stage_col].value_counts()

    # Prepare data
    stages = stage_counts.index.astype(str).tolist()
    counts = stage_counts.values.tolist()

    # Calculate percentages for pie chart
    angles = [c/sum(counts) * 2*np.pi for c in counts]
    colors_list = Category20[20][:len(stages)] if len(
        stages) <= 20 else Turbo256[::256//len(stages)][:len(stages)]

    # Create source
    data = {
        'stage': stages,
        'count': counts,
        'angle': angles,
        'color': colors_list,
        'percentage': [f"{c/sum(counts)*100:.1f}%" for c in counts]
    }
    source = ColumnDataSource(data)

    # Pie chart using wedge
    p5 = figure(
        width=500,
        height=500,
        title="AJCC Pathologic Stage Distribution",
        toolbar_location="right",
        tools="hover,save",
        tooltips="@stage: @count (@percentage)",
        x_range=(-1.5, 1.5),
        y_range=(-1.5, 1.5)
    )

    p5.wedge(
        x=0, y=0, radius=1,
        start_angle=cumsum('angle', include_zero=True),
        end_angle=cumsum('angle'),
        line_color='white',
        fill_color='color',
        legend_field='stage',
        source=source
    )

    p5.axis.visible = False
    p5.grid.visible = False
    p5.legend.location = "center_right"
    p5.legend.label_text_font_size = '9pt'

    # Bar chart alternative
    p6 = figure(
        x_range=stages,
        width=900,
        height=400,
        title="AJCC Pathologic Stage - Bar Chart",
        toolbar_location="right",
        tools="pan,wheel_zoom,box_zoom,reset,save"
    )

    p6.vbar(
        x='stage',
        top='count',
        width=0.8,
        source=source,
        fill_color='color',
        line_color='white',
        alpha=0.8
    )

    p6.xaxis.axis_label = "Stage"
    p6.yaxis.axis_label = "Number of Patients"
    p6.xaxis.major_label_orientation = 0.785  # 45 degrees

    hover = HoverTool(tooltips=[
        ("Stage", "@stage"),
        ("Count", "@count"),
        ("Percentage", "@percentage")
    ])
    p6.add_tools(hover)

    show(row(p5, p6))

In [14]:
# TNM staging components comparison
tnm_cols = ['ajcc_pathologic_t.diagnoses',
            'ajcc_pathologic_n.diagnoses', 'ajcc_pathologic_m.diagnoses']
tnm_names = ['T Stage', 'N Stage', 'M Stage']

plots = []

for col, name in zip(tnm_cols, tnm_names):
    if col in demo_df.columns:
        tnm_counts = demo_df[col].value_counts().head(15)  # Top 15 values

        tnm_data = pd.DataFrame({
            'stage': tnm_counts.index.astype(str),
            'count': tnm_counts.values
        })

        p = figure(
            y_range=tnm_data['stage'].tolist(),
            width=500,
            height=400,
            title=f"{name} Distribution",
            toolbar_location="right",
            tools="pan,wheel_zoom,box_zoom,reset,save"
        )

        p.hbar(
            y='stage',
            right='count',
            height=0.8,
            source=ColumnDataSource(tnm_data),
            fill_color=factor_cmap(
                'stage', palette=Viridis256, factors=tnm_data['stage'].tolist()),
            alpha=0.8
        )

        p.xaxis.axis_label = "Count"

        hover = HoverTool(tooltips=[
            ("Stage", "@stage"),
            ("Count", "@count")
        ])
        p.add_tools(hover)

        plots.append(p)

if plots:
    show(row(*plots))

---
## Part 5: Disease Characteristics

Analysis of primary diagnosis, morphology, tissue origin, and tumor classification.

In [15]:
# Primary diagnosis analysis
primary_dx_col = 'primary_diagnosis.diagnoses'

if primary_dx_col in demo_df.columns:
    primary_dx_counts = demo_df[primary_dx_col].value_counts()

    print("Primary Diagnosis Distribution:")
    print(f"  Unique diagnoses: {len(primary_dx_counts)}")
    print(f"  Missing: {demo_df[primary_dx_col].isnull().sum()}")
    print(f"\nTop 15 diagnoses:")
    for i, (dx, count) in enumerate(primary_dx_counts.head(15).items(), 1):
        print(f"  {i:2d}. {dx}: {count} ({count/len(demo_df)*100:.1f}%)")

Primary Diagnosis Distribution:
  Unique diagnoses: 22
  Missing: 3

Top 15 diagnoses:
   1. Infiltrating duct carcinoma, NOS: 899 (71.5%)
   2. Lobular carcinoma, NOS: 218 (17.3%)
   3. Infiltrating duct and lobular carcinoma: 38 (3.0%)
   4. Infiltrating duct mixed with other types of carcinoma: 21 (1.7%)
   5. Metaplastic carcinoma, NOS: 17 (1.4%)
   6. Mucinous adenocarcinoma: 16 (1.3%)
   7. Infiltrating lobular mixed with other types of carcinoma: 8 (0.6%)
   8. Medullary carcinoma, NOS: 8 (0.6%)
   9. Intraductal papillary adenocarcinoma with invasion: 6 (0.5%)
  10. Intraductal micropapillary carcinoma: 4 (0.3%)
  11. Paget disease and infiltrating duct carcinoma of breast: 4 (0.3%)
  12. Pleomorphic carcinoma: 3 (0.2%)
  13. Papillary carcinoma, NOS: 2 (0.2%)
  14. Phyllodes tumor, malignant: 2 (0.2%)
  15. Adenoid cystic carcinoma: 1 (0.1%)


In [16]:
# Visualize top primary diagnoses
if primary_dx_col in demo_df.columns:
    top_dx = demo_df[primary_dx_col].value_counts().head(20)

    # Shorten diagnosis names for better display
    short_dx_names = [
        dx[:60] + '...' if len(dx) > 60 else dx for dx in top_dx.index]

    dx_data = pd.DataFrame({
        'diagnosis': short_dx_names,
        'full_diagnosis': top_dx.index.tolist(),
        'count': top_dx.values,
        'percentage': [f"{c/len(demo_df)*100:.1f}%" for c in top_dx.values]
    })

    source = ColumnDataSource(dx_data)

    p7 = figure(
        y_range=short_dx_names,
        width=900,
        height=600,
        title="Top 20 Primary Diagnoses",
        toolbar_location="right",
        tools="pan,wheel_zoom,box_zoom,reset,save"
    )

    p7.hbar(
        y='diagnosis',
        right='count',
        height=0.8,
        source=source,
        fill_color='#e74c3c',
        alpha=0.8
    )

    p7.xaxis.axis_label = "Number of Patients"

    hover = HoverTool(tooltips=[
        ("Diagnosis", "@full_diagnosis"),
        ("Count", "@count"),
        ("Percentage", "@percentage")
    ])
    p7.add_tools(hover)

    show(p7)

In [17]:
# Morphology analysis
morph_col = 'morphology.diagnoses'

if morph_col in demo_df.columns:
    morph_counts = demo_df[morph_col].value_counts().head(15)

    print("\nMorphology Distribution (Top 15):")
    for morph, count in morph_counts.items():
        print(f"  {morph}: {count} ({count/len(demo_df)*100:.1f}%)")

    # Visualize
    morph_data = pd.DataFrame({
        'morphology': morph_counts.index.astype(str),
        'count': morph_counts.values
    })

    p8 = figure(
        x_range=morph_data['morphology'].tolist(),
        width=900,
        height=400,
        title="Morphology Distribution",
        toolbar_location="right",
        tools="pan,wheel_zoom,box_zoom,reset,save"
    )

    p8.vbar(
        x='morphology',
        top='count',
        width=0.8,
        source=ColumnDataSource(morph_data),
        fill_color='#3498db',
        line_color='white',
        alpha=0.8
    )

    p8.xaxis.axis_label = "Morphology Code"
    p8.yaxis.axis_label = "Count"
    p8.xaxis.major_label_orientation = 0.785

    show(p8)


Morphology Distribution (Top 15):
  8500/3: 899 (71.5%)
  8520/3: 218 (17.3%)
  8522/3: 38 (3.0%)
  8523/3: 21 (1.7%)
  8575/3: 17 (1.4%)
  8480/3: 16 (1.3%)
  8524/3: 8 (0.6%)
  8510/3: 8 (0.6%)
  8503/3: 6 (0.5%)
  8507/3: 4 (0.3%)
  8541/3: 4 (0.3%)
  8022/3: 3 (0.2%)
  8050/3: 2 (0.2%)
  9020/3: 2 (0.2%)
  8200/3: 1 (0.1%)


In [18]:
# Tissue/organ of origin
tissue_col = 'tissue_or_organ_of_origin.diagnoses'

if tissue_col in demo_df.columns:
    tissue_counts = demo_df[tissue_col].value_counts()

    print("\nTissue/Organ of Origin:")
    print(f"  Unique tissues: {len(tissue_counts)}")
    print(f"\nDistribution:")
    for tissue, count in tissue_counts.items():
        print(f"  {tissue}: {count} ({count/len(demo_df)*100:.1f}%)")

    if len(tissue_counts) > 0:
        # Pie chart
        data = {
            'tissue': tissue_counts.index.astype(str).tolist(),
            'count': tissue_counts.values.tolist(),
            'angle': [c/sum(tissue_counts) * 2*np.pi for c in tissue_counts.values],
            'color': Category20[20][:len(tissue_counts)],
            'percentage': [f"{c/sum(tissue_counts)*100:.1f}%" for c in tissue_counts.values]
        }
        source = ColumnDataSource(data)

        p9 = figure(
            width=700,
            height=500,
            title="Tissue/Organ of Origin Distribution",
            toolbar_location="right",
            tools="hover,save",
            tooltips="@tissue: @count (@percentage)",
            x_range=(-1.5, 1.5),
            y_range=(-1.5, 1.5)
        )

        p9.wedge(
            x=0, y=0, radius=1,
            start_angle=cumsum('angle', include_zero=True),
            end_angle=cumsum('angle'),
            line_color='white',
            fill_color='color',
            legend_field='tissue',
            source=source
        )

        p9.axis.visible = False
        p9.grid.visible = False
        p9.legend.location = "center_right"

        show(p9)


Tissue/Organ of Origin:
  Unique tissues: 6

Distribution:
  Breast, NOS: 1237 (98.4%)
  Lower-inner quadrant of breast: 6 (0.5%)
  Upper-outer quadrant of breast: 5 (0.4%)
  Overlapping lesion of breast: 3 (0.2%)
  Upper-inner quadrant of breast: 2 (0.2%)
  Lower-outer quadrant of breast: 1 (0.1%)


---
## Part 6: Tumor Classification & Grading

In [19]:
# Tumor grade analysis
grade_col = 'tumor_grade.diagnoses'

if grade_col in demo_df.columns:
    grade_counts = demo_df[grade_col].value_counts()

    print("Tumor Grade Distribution:")
    for grade, count in grade_counts.items():
        print(f"  {grade}: {count} ({count/len(demo_df)*100:.1f}%)")

    # Visualization
    grade_data = pd.DataFrame({
        'grade': grade_counts.index.astype(str),
        'count': grade_counts.values,
        'percentage': [f"{c/len(demo_df)*100:.1f}%" for c in grade_counts.values]
    })

    source = ColumnDataSource(grade_data)

    p10 = figure(
        x_range=grade_data['grade'].tolist(),
        width=900,
        height=400,
        title="Tumor Grade Distribution",
        toolbar_location="right",
        tools="pan,wheel_zoom,box_zoom,reset,save"
    )

    colors = ['#2ecc71', '#f39c12', '#e74c3c', '#9b59b6'][:len(grade_data)]

    p10.vbar(
        x='grade',
        top='count',
        width=0.8,
        source=source,
        fill_color=factor_cmap('grade', palette=colors,
                               factors=grade_data['grade'].tolist()),
        line_color='white',
        alpha=0.8
    )

    p10.xaxis.axis_label = "Tumor Grade"
    p10.yaxis.axis_label = "Number of Patients"

    hover = HoverTool(tooltips=[
        ("Grade", "@grade"),
        ("Count", "@count"),
        ("Percentage", "@percentage")
    ])
    p10.add_tools(hover)

    show(p10)

Tumor Grade Distribution:
  Not Reported: 1254 (99.8%)


In [20]:
# Classification of tumor
class_col = 'classification_of_tumor.diagnoses'

if class_col in demo_df.columns:
    class_counts = demo_df[class_col].value_counts()

    print("\nClassification of Tumor:")
    for cls, count in class_counts.items():
        print(f"  {cls}: {count} ({count/len(demo_df)*100:.1f}%)")

    if len(class_counts) > 0:
        class_data = pd.DataFrame({
            'classification': class_counts.index.astype(str),
            'count': class_counts.values,
            'angle': [c/sum(class_counts) * 2*np.pi for c in class_counts.values],
            'color': Spectral11[:len(class_counts)],
            'percentage': [f"{c/sum(class_counts)*100:.1f}%" for c in class_counts.values]
        })

        source = ColumnDataSource(class_data)

        p11 = figure(
            width=700,
            height=500,
            title="Tumor Classification Distribution",
            toolbar_location="right",
            tools="hover,save",
            tooltips="@classification: @count (@percentage)",
            x_range=(-1.5, 1.5),
            y_range=(-1.5, 1.5)
        )

        p11.wedge(
            x=0, y=0, radius=1,
            start_angle=cumsum('angle', include_zero=True),
            end_angle=cumsum('angle'),
            line_color='white',
            fill_color='color',
            legend_field='classification',
            source=source
        )

        p11.axis.visible = False
        p11.grid.visible = False
        p11.legend.location = "center_right"

        show(p11)


Classification of Tumor:
  not reported: 1254 (99.8%)


---
## Part 7: Prior Medical History

Analysis of prior malignancy, prior treatment, and synchronous malignancy.

In [21]:
# Prior malignancy, prior treatment, synchronous malignancy
history_cols = {
    'Prior Malignancy': 'prior_malignancy.diagnoses',
    'Prior Treatment': 'prior_treatment.diagnoses',
    'Synchronous Malignancy': 'synchronous_malignancy.diagnoses'
}

history_data = {}

for name, col in history_cols.items():
    if col in demo_df.columns:
        counts = demo_df[col].value_counts()
        history_data[name] = counts
        print(f"\n{name}:")
        for val, count in counts.items():
            print(f"  {val}: {count} ({count/len(demo_df)*100:.1f}%)")


Prior Malignancy:
  no: 1176 (93.6%)
  yes: 77 (6.1%)
  not reported: 1 (0.1%)

Prior Treatment:
  No: 1238 (98.5%)
  Yes: 14 (1.1%)
  Not Reported: 2 (0.2%)

Synchronous Malignancy:
  No: 1176 (93.6%)
  Not Reported: 78 (6.2%)


In [22]:
# Visualize medical history as grouped bar chart
if history_data:
    # Prepare data for grouped bars
    categories = []
    yes_counts = []
    no_counts = []

    for name, counts in history_data.items():
        categories.append(name)
        yes_counts.append(counts.get('Yes', counts.get('yes', 0)))
        no_counts.append(counts.get('No', counts.get(
            'no', counts.get('Not Reported', 0))))

    source = ColumnDataSource(data=dict(
        categories=categories,
        yes=yes_counts,
        no=no_counts
    ))

    p12 = figure(
        x_range=categories,
        width=900,
        height=400,
        title="Prior Medical History Summary",
        toolbar_location="right",
        tools="pan,wheel_zoom,box_zoom,reset,save"
    )

    p12.vbar(x=dodge('categories', -0.15, range=p12.x_range), top='yes', width=0.3,
             source=source, color='#e74c3c', legend_label="Yes", alpha=0.8)
    p12.vbar(x=dodge('categories', 0.15, range=p12.x_range), top='no', width=0.3,
             source=source, color='#2ecc71', legend_label="No/Not Reported", alpha=0.8)

    p12.xaxis.axis_label = "Category"
    p12.yaxis.axis_label = "Number of Patients"
    p12.xaxis.major_label_orientation = 0.785
    p12.legend.location = "top_right"

    from bokeh.models import Dodge

    show(p12)

NameError: name 'dodge' is not defined

---
## Part 8: Disease Status & Outcomes

In [ ]:
# Last known disease status
status_col = 'last_known_disease_status.diagnoses'

if status_col in demo_df.columns:
    status_counts = demo_df[status_col].value_counts()

    print("Last Known Disease Status:")
    for status, count in status_counts.items():
        print(f"  {status}: {count} ({count/len(demo_df)*100:.1f}%)")

    # Visualization
    status_data = pd.DataFrame({
        'status': status_counts.index.astype(str),
        'count': status_counts.values,
        'angle': [c/sum(status_counts) * 2*np.pi for c in status_counts.values],
        'percentage': [f"{c/sum(status_counts)*100:.1f}%" for c in status_counts.values]
    })

    # Color mapping based on status
    color_map = {
        'Tumor Free': '#2ecc71',
        'With Tumor': '#e74c3c',
        'Not Reported': '#95a5a6'
    }
    status_data['color'] = [color_map.get(
        s, '#3498db') for s in status_data['status']]

    source = ColumnDataSource(status_data)

    p13 = figure(
        width=700,
        height=500,
        title="Last Known Disease Status",
        toolbar_location="right",
        tools="hover,save",
        tooltips="@status: @count (@percentage)",
        x_range=(-1.5, 1.5),
        y_range=(-1.5, 1.5)
    )

    p13.wedge(
        x=0, y=0, radius=1,
        start_angle=cumsum('angle', include_zero=True),
        end_angle=cumsum('angle'),
        line_color='white',
        fill_color='color',
        legend_field='status',
        source=source
    )

    p13.axis.visible = False
    p13.grid.visible = False
    p13.legend.location = "center_right"

    show(p13)

In [ ]:
# Progression or recurrence
prog_col = 'progression_or_recurrence.diagnoses'

if prog_col in demo_df.columns:
    prog_counts = demo_df[prog_col].value_counts()

    print("\nProgression or Recurrence:")
    for val, count in prog_counts.items():
        print(f"  {val}: {count} ({count/len(demo_df)*100:.1f}%)")

    # Bar chart
    prog_data = pd.DataFrame({
        'category': prog_counts.index.astype(str),
        'count': prog_counts.values,
        'percentage': [f"{c/len(demo_df)*100:.1f}%" for c in prog_counts.values]
    })

    source = ColumnDataSource(prog_data)

    p14 = figure(
        x_range=prog_data['category'].tolist(),
        width=900,
        height=400,
        title="Progression or Recurrence Status",
        toolbar_location="right",
        tools="pan,wheel_zoom,box_zoom,reset,save"
    )

    p14.vbar(
        x='category',
        top='count',
        width=0.8,
        source=source,
        fill_color=factor_cmap('category', palette=['#e74c3c', '#2ecc71', '#95a5a6'],
                               factors=prog_data['category'].tolist()),
        line_color='white',
        alpha=0.8
    )

    p14.xaxis.axis_label = "Status"
    p14.yaxis.axis_label = "Number of Patients"

    hover = HoverTool(tooltips=[
        ("Status", "@category"),
        ("Count", "@count"),
        ("Percentage", "@percentage")
    ])
    p14.add_tools(hover)

    show(p14)

---
## Part 9: Treatment Information

Analysis of treatment types and therapy patterns.

In [ ]:
# Treatment-related columns
treatment_cols = {
    'Treatment Type': 'treatment_type.treatments.diagnoses',
    'Treatment or Therapy': 'treatment_or_therapy.treatments.diagnoses',
    'State': 'state.treatments.diagnoses'
}

for name, col in treatment_cols.items():
    if col in demo_df.columns:
        counts = demo_df[col].value_counts()
        print(f"\n{name}:")
        print(f"  Unique values: {len(counts)}")
        print(
            f"  Missing: {demo_df[col].isnull().sum()} ({demo_df[col].isnull().sum()/len(demo_df)*100:.1f}%)")
        print(f"  Distribution:")
        for val, count in counts.head(10).items():
            print(
                f"    {val}: {count} ({count/demo_df[col].notnull().sum()*100:.1f}% of non-missing)")

In [ ]:
# Visualize treatment types
treatment_type_col = 'treatment_type.treatments.diagnoses'

if treatment_type_col in demo_df.columns:
    treatment_counts = demo_df[treatment_type_col].value_counts().head(15)

    treatment_data = pd.DataFrame({
        'treatment': treatment_counts.index.astype(str),
        'count': treatment_counts.values,
        'percentage': [f"{c/demo_df[treatment_type_col].notnull().sum()*100:.1f}%"
                       for c in treatment_counts.values]
    })

    source = ColumnDataSource(treatment_data)

    p15 = figure(
        y_range=treatment_data['treatment'].tolist(),
        width=900,
        height=500,
        title="Top Treatment Types",
        toolbar_location="right",
        tools="pan,wheel_zoom,box_zoom,reset,save"
    )

    p15.hbar(
        y='treatment',
        right='count',
        height=0.8,
        source=source,
        fill_color='#9b59b6',
        alpha=0.8
    )

    p15.xaxis.axis_label = "Number of Patients"

    hover = HoverTool(tooltips=[
        ("Treatment", "@treatment"),
        ("Count", "@count"),
        ("Percentage", "@percentage")
    ])
    p15.add_tools(hover)

    show(p15)

---
## Part 10: Cross-Variable Comparisons

Comparing key demographic variables against each other.

In [ ]:
# Age vs Stage comparison
if 'age_at_diagnosis.diagnoses' in demo_df.columns and 'ajcc_pathologic_stage.diagnoses' in demo_df.columns:

    # Prepare data
    comparison_df = demo_df[['age_at_diagnosis.diagnoses',
                             'ajcc_pathologic_stage.diagnoses']].copy()
    comparison_df['age'] = pd.to_numeric(
        comparison_df['age_at_diagnosis.diagnoses'], errors='coerce')
    comparison_df = comparison_df.dropna()

    # Group by stage
    stage_groups = comparison_df.groupby('ajcc_pathologic_stage.diagnoses')[
        'age'].agg(['mean', 'median', 'std', 'count'])
    stage_groups = stage_groups.sort_values('mean', ascending=False)

    print("Age Statistics by AJCC Pathologic Stage:")
    print(stage_groups.to_string())

    # Box plot style visualization
    stages = stage_groups.index.astype(str).tolist()

    # Calculate quartiles for each stage
    q1s = []
    medians = []
    q3s = []
    means = []

    for stage in stages:
        stage_ages = comparison_df[comparison_df['ajcc_pathologic_stage.diagnoses'] == stage]['age']
        q1s.append(stage_ages.quantile(0.25))
        medians.append(stage_ages.median())
        q3s.append(stage_ages.quantile(0.75))
        means.append(stage_ages.mean())

    p16 = figure(
        x_range=stages,
        width=900,
        height=500,
        title="Age Distribution by AJCC Pathologic Stage",
        toolbar_location="right",
        tools="pan,wheel_zoom,box_zoom,reset,save"
    )

    # Plot median as bars
    p16.vbar(x=stages, top=medians, width=0.6,
             fill_color='#3498db', alpha=0.6, legend_label='Median Age')

    # Plot means as circles
    p16.circle(stages, means, size=10, color='#e74c3c',
               alpha=0.8, legend_label='Mean Age')

    # Add error bars (Q1 to Q3)
    for i, stage in enumerate(stages):
        p16.line([i, i], [q1s[i], q3s[i]],
                 line_width=2, color='black', alpha=0.5)

    p16.xaxis.axis_label = "AJCC Pathologic Stage"
    p16.yaxis.axis_label = "Age at Diagnosis (years)"
    p16.xaxis.major_label_orientation = 0.785
    p16.legend.location = "top_right"

    show(p16)

In [ ]:
# Grade vs Stage comparison (heatmap-style)
if 'tumor_grade.diagnoses' in demo_df.columns and 'ajcc_pathologic_stage.diagnoses' in demo_df.columns:

    # Create crosstab
    crosstab = pd.crosstab(
        demo_df['tumor_grade.diagnoses'],
        demo_df['ajcc_pathologic_stage.diagnoses']
    )

    print("\nTumor Grade vs AJCC Stage Cross-tabulation:")
    print(crosstab)

    # Prepare data for heatmap
    from bokeh.models import LinearColorMapper
    from bokeh.transform import transform

    # Convert to long format
    grades = []
    stages = []
    counts = []

    for grade in crosstab.index:
        for stage in crosstab.columns:
            grades.append(str(grade))
            stages.append(str(stage))
            counts.append(crosstab.loc[grade, stage])

    source = ColumnDataSource(data=dict(
        grade=grades,
        stage=stages,
        count=counts
    ))

    # Create color mapper
    mapper = LinearColorMapper(
        palette=Viridis256, low=min(counts), high=max(counts))

    p17 = figure(
        x_range=sorted(list(set(stages))),
        y_range=sorted(list(set(grades))),
        width=900,
        height=500,
        title="Tumor Grade vs AJCC Stage Heatmap",
        toolbar_location="right",
        tools="hover,save",
        tooltips=[("Grade", "@grade"), ("Stage", "@stage"),
                  ("Count", "@count")]
    )

    p17.rect(
        x='stage',
        y='grade',
        width=1,
        height=1,
        source=source,
        fill_color=transform('count', mapper),
        line_color='white'
    )

    p17.xaxis.axis_label = "AJCC Pathologic Stage"
    p17.yaxis.axis_label = "Tumor Grade"
    p17.xaxis.major_label_orientation = 0.785

    # Add color bar
    from bokeh.models import ColorBar
    color_bar = ColorBar(color_mapper=mapper, width=8, location=(0, 0))
    p17.add_layout(color_bar, 'right')

    show(p17)

---
## Part 11: ICD-10 Codes & Site of Resection

In [ ]:
# ICD-10 code analysis
icd_col = 'icd_10_code.diagnoses'

if icd_col in demo_df.columns:
    icd_counts = demo_df[icd_col].value_counts()

    print("ICD-10 Code Distribution:")
    print(f"  Unique codes: {len(icd_counts)}")
    print(
        f"  Missing: {demo_df[icd_col].isnull().sum()} ({demo_df[icd_col].isnull().sum()/len(demo_df)*100:.1f}%)")
    print(f"\nTop 10 codes:")
    for code, count in icd_counts.head(10).items():
        print(f"  {code}: {count} ({count/len(demo_df)*100:.1f}%)")

In [ ]:
# Site of resection or biopsy
site_col = 'site_of_resection_or_biopsy.diagnoses'

if site_col in demo_df.columns:
    site_counts = demo_df[site_col].value_counts()

    print("\nSite of Resection or Biopsy:")
    print(f"  Unique sites: {len(site_counts)}")
    print(f"  Missing: {demo_df[site_col].isnull().sum()}")
    print(f"\nDistribution:")
    for site, count in site_counts.items():
        print(f"  {site}: {count} ({count/len(demo_df)*100:.1f}%)")

    if len(site_counts) > 0 and len(site_counts) <= 20:
        # Visualize
        site_data = pd.DataFrame({
            'site': site_counts.index.astype(str),
            'count': site_counts.values,
            'percentage': [f"{c/len(demo_df)*100:.1f}%" for c in site_counts.values]
        })

        source = ColumnDataSource(site_data)

        p18 = figure(
            y_range=site_data['site'].tolist(),
            width=900,
            height=max(400, len(site_counts) * 30),
            title="Site of Resection or Biopsy Distribution",
            toolbar_location="right",
            tools="pan,wheel_zoom,box_zoom,reset,save"
        )

        p18.hbar(
            y='site',
            right='count',
            height=0.8,
            source=source,
            fill_color='#16a085',
            alpha=0.8
        )

        p18.xaxis.axis_label = "Number of Patients"

        hover = HoverTool(tooltips=[
            ("Site", "@site"),
            ("Count", "@count"),
            ("Percentage", "@percentage")
        ])
        p18.add_tools(hover)

        show(p18)

---
## Part 12: Summary Statistics Table

Comprehensive summary of all demographic variables.

In [ ]:
# Create comprehensive summary table
summary_data = []

for col in existing_cols:
    col_data = demo_df[col]

    # Basic stats
    total = len(col_data)
    missing = col_data.isnull().sum()
    present = total - missing
    unique = col_data.nunique()

    # Try to get numeric stats if applicable
    numeric_data = pd.to_numeric(col_data, errors='coerce')
    if numeric_data.notnull().sum() > 0:
        mean_val = numeric_data.mean()
        median_val = numeric_data.median()
        std_val = numeric_data.std()
        data_type = 'Numeric'
    else:
        mean_val = None
        median_val = None
        std_val = None
        data_type = 'Categorical'

    # Most common value
    if present > 0:
        mode_val = col_data.mode().values[0] if len(
            col_data.mode()) > 0 else None
        mode_count = col_data.value_counts().iloc[0] if len(
            col_data.value_counts()) > 0 else 0
    else:
        mode_val = None
        mode_count = 0

    summary_data.append({
        'Column': col.replace('.diagnoses', '').replace('.xena_derived', '').replace('.treatments', ''),
        'Type': data_type,
        'Total': total,
        'Present': present,
        'Missing': missing,
        'Missing %': f"{missing/total*100:.1f}",
        'Unique': unique,
        'Mode': str(mode_val)[:30] if mode_val else 'N/A',
        'Mode Count': mode_count,
        'Mean': f"{mean_val:.2f}" if mean_val is not None else 'N/A',
        'Median': f"{median_val:.2f}" if median_val is not None else 'N/A'
    })

summary_df = pd.DataFrame(summary_data)
summary_df = summary_df.sort_values('Missing %', ascending=False)

print("\nComprehensive Summary Statistics:")
print("="*100)
print(summary_df.to_string(index=False))

---
## Conclusions

This comprehensive EDA has covered all 30 demographic and diagnosis-related columns:

### Key Findings:

1. **Data Quality**: Identified columns with high/low completeness
2. **Age Patterns**: Analyzed age distribution at diagnosis and earliest diagnosis
3. **Temporal Trends**: Examined diagnosis years and follow-up times
4. **AJCC Staging**: Comprehensive TNM staging analysis
5. **Disease Characteristics**: Primary diagnosis, morphology, and tissue origin distributions
6. **Tumor Classification**: Grade and classification patterns
7. **Medical History**: Prior malignancy, treatment, and synchronous malignancy
8. **Treatment Patterns**: Treatment types and therapy information
9. **Outcomes**: Disease status and progression/recurrence
10. **Cross-Comparisons**: Age vs stage, grade vs stage, and other correlations

### Visualization Highlights:
- Interactive Bokeh plots for all major variables
- Hover tooltips for detailed information
- Multiple chart types (histograms, bar charts, pie charts, heatmaps)
- Color-coded visualizations for easy interpretation

In [ ]:
print("\n" + "="*80)
print("Analysis Complete!")
print("="*80)
print(f"Total patients analyzed: {len(demo_df):,}")
print(f"Total demographic columns analyzed: {len(existing_cols)}")
print(f"Visualizations created: 18+ interactive Bokeh charts")

In [ ]:
# Grade vs Stage comparison (heatmap-style)
if 'tumor_grade.diagnoses' in demo_df.columns and 'ajcc_pathologic_stage.diagnoses' in demo_df.columns:

    # Create crosstab
    crosstab = pd.crosstab(
        demo_df['tumor_grade.diagnoses'],
        demo_df['ajcc_pathologic_stage.diagnoses']
    )

    print("\nTumor Grade vs AJCC Stage Cross-tabulation:")
    print(crosstab)

    # Prepare data for heatmap
    from bokeh.models import LinearColorMapper
    from bokeh.transform import transform

    # Convert to long format
    grades = []
    stages = []
    counts = []

    for grade in crosstab.index:
        for stage in crosstab.columns:
            grades.append(str(grade))
            stages.append(str(stage))
            counts.append(crosstab.loc[grade, stage])

    source = ColumnDataSource(data=dict(
        grade=grades,
        stage=stages,
        count=counts
    ))

    # Create color mapper
    mapper = LinearColorMapper(
        palette=Viridis256, low=min(counts), high=max(counts))

    p17 = figure(
        x_range=sorted(list(set(stages))),
        y_range=sorted(list(set(grades))),
        width=900,
        height=500,
        title="Tumor Grade vs AJCC Stage Heatmap",
        toolbar_location="right",
        tools="hover,save",
        tooltips=[("Grade", "@grade"), ("Stage", "@stage"),
                  ("Count", "@count")]
    )

    p17.rect(
        x='stage',
        y='grade',
        width=1,
        height=1,
        source=source,
        fill_color=transform('count', mapper),
        line_color='white'
    )

    p17.xaxis.axis_label = "AJCC Pathologic Stage"
    p17.yaxis.axis_label = "Tumor Grade"
    p17.xaxis.major_label_orientation = 0.785

    # Add color bar
    from bokeh.models import ColorBar
    color_bar = ColorBar(color_mapper=mapper, width=8, location=(0, 0))
    p17.add_layout(color_bar, 'right')

    show(p17)